# ⚖️ LexCrisis — SFT + Curriculum Training Notebook

**OpenEnv Hackathon | Meta × PyTorch × Hugging Face × Scaler**

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** on LexCrisis legal-ops trajectories using:
- **Phase 1** — Supervised Fine-Tuning (SFT) on oracle-reference trajectories
- **Phase 2** — Seeded curriculum episodes for deadline and ordering robustness
- **Phase 3** — Artifact-backed evaluation with `collect_traces.py` and `evaluate_runs.py`

It intentionally avoids hardcoded result claims. Treat `outputs/evals/*.json` and `assets/*.png`
as the source of truth for before/after comparisons.

> **Runtime target:** Free Colab T4 or Hugging Face T4 small  
> **Memory target:** ~6 GB VRAM (4-bit quantised LoRA)

## 0. Install Dependencies

## 0b. Initialize Experiment Tracking (W&B)

W&B logs every training step, loss curve, reward signals, and final eval metrics automatically.
Get a free API key at https://wandb.ai/authorize

Set `USE_WANDB = False` below to skip tracking.

In [ ]:
import wandb

USE_WANDB = True  # Set False to disable

if USE_WANDB:
    wandb.login()  # prompts for API key on first run
    wandb.init(
        project="lexcrisis-sft",
        name="qwen2.5-1.5b-sft-colab",
        config={
            "model": "unsloth/Qwen2.5-1.5B-Instruct",
            "lora_rank": 16,
            "lora_alpha": 16,
            "batch_size": 2,
            "gradient_accumulation": 4,
            "learning_rate": 2e-4,
            "epochs": 3,
            "environment": "lexcrisis",
            "tasks": "task_1,task_2,task_3",
        },
        tags=["sft", "lexcrisis", "openenv", "hackathon"],
    )
    print(f"W&B run: {wandb.run.url}")
else:
    print("W&B tracking disabled.")

In [ ]:
# Unsloth — 2x faster fine-tuning with 40% less VRAM
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl transformers datasets peft accelerate bitsandbytes -q
# LexCrisis environment dependencies
!pip install openenv-core>=0.1.13 fastapi uvicorn pydantic openai requests -q
!pip install matplotlib numpy wandb -q

## 1. Clone LexCrisis Environment

In [ ]:
import os, subprocess, json
from pathlib import Path

# Clone the submitted LexCrisis repository
REPO_URL = "https://github.com/RadheRadheontop/LexCrisis.git"

if not Path("lexcrisis").exists():
    result = subprocess.run(["git", "clone", REPO_URL, "lexcrisis"],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print("Already cloned.")

os.chdir("lexcrisis")
print("Working directory:", os.getcwd())
print("Files:", [f for f in os.listdir() if not f.startswith('.')])

## 2. Phase 1 — Generate SFT Data from Oracle-Reference Trajectories

In [ ]:
# Run the self_improve pipeline to generate ShareGPT-format training examples
result = subprocess.run(
    ["python", "self_improve.py", "--phase", "sft"],
    capture_output=True, text=True
)
print(result.stderr)  # progress is written to stderr
if result.returncode != 0:
    print("STDOUT:", result.stdout)

sft_path = Path("outputs/self_improve/sft_examples.jsonl")
assert sft_path.exists(), f"SFT file not found at {sft_path}"

sft_data = [json.loads(l) for l in sft_path.read_text().splitlines() if l.strip()]
print(f"\nLoaded {len(sft_data)} SFT examples across 3 tasks")

# Inspect one example
ex = sft_data[0]
print(f"\nSample (task={ex['metadata']['task_id']}, step={ex['metadata']['step']}):")
for turn in ex['conversations']:
    print(f"  [{turn['from']}]: {turn['value'][:120]}...")

In [ ]:
# Also generate adversarial curriculum episodes (Phase 2)
result = subprocess.run(
    ["python", "self_improve.py", "--phase", "curriculum", "--variants", "10"],
    capture_output=True, text=True
)
print(result.stderr)

curriculum_path = Path("outputs/self_improve/curriculum_episodes.jsonl")
curriculum_data = [json.loads(l) for l in curriculum_path.read_text().splitlines() if l.strip()]
print(f"Generated {len(curriculum_data)} curriculum episodes")

## 3. Load Qwen2.5-1.5B-Instruct with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
DTYPE = None          # auto-detect: bfloat16 on Ampere+, float16 on T4
LOAD_IN_4BIT = True   # 4-bit NF4 quantisation — fits in free T4 VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

# Apply LoRA adapters — only ~1% of parameters are trainable
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 4. Format Dataset with Chat Template

In [ ]:
from datasets import Dataset

def format_example(example: dict) -> dict:
    """Convert ShareGPT format to a single tokenized string."""
    convs = example["conversations"]
    messages = [
        {"role": "system",    "content": convs[0]["value"]},
        {"role": "user",      "content": convs[1]["value"]},
        {"role": "assistant", "content": convs[2]["value"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

raw_dataset = Dataset.from_list(sft_data)
dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)

print(f"Dataset: {len(dataset)} training examples")
print("\nSample text (first 400 chars):")
print(dataset[0]["text"][:400])

## 5. Evaluate Baseline (Before Fine-tuning)

In [ ]:
# Measure oracle-reference scores before any training
import sys
sys.path.insert(0, ".")

from lexcrisis_env.env import LexCrisisEngine
from lexcrisis_env.models import Action
from lexcrisis_env.tasks import SCRIPTED_BASELINES, TASK_DEFINITIONS

def run_oracle_reference(task_id: str) -> float:
    engine = LexCrisisEngine()
    engine.reset(task_id=task_id)
    for raw in SCRIPTED_BASELINES[task_id]:
        _, _, done, _ = engine.step(Action.model_validate(raw))
        if done:
            break
    return engine.last_score

baseline_scores = {tid: run_oracle_reference(tid) for tid in ["task_1", "task_2", "task_3"]}
print("Oracle reference scores:")
for tid, score in baseline_scores.items():
    print(f"  {tid}: {score:.4f}")

## 6. Fine-tune with TRL SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs/training",
        report_to="wandb" if USE_WANDB else "none",  # experiment tracking
    ),
)

print("Starting SFT training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Steps:          {trainer_stats.global_step}")
print(f"  Training time:  {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Final loss:     {trainer_stats.metrics['train_loss']:.4f}")

## 7. Plot Training Loss

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
BG, AX, GR = "#0F172A", "#1E293B", "#334155"
plt.rcParams.update({
    "axes.facecolor": AX, "figure.facecolor": BG, "axes.edgecolor": GR,
    "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white",
    "text.color": "white", "grid.color": "#475569", "grid.alpha": 0.15,
    "legend.facecolor": AX, "legend.edgecolor": GR,
})

log_history = trainer.state.log_history
train_logs  = [x for x in log_history if "loss" in x]
steps  = [x["step"] for x in train_logs]
losses = [x["loss"]  for x in train_logs]

def moving_avg(y, w=5):
    k = np.ones(w) / w
    p = np.pad(y, (w // 2, w // 2), mode="edge")
    return np.convolve(p, k, mode="valid")[:len(y)]

sm = moving_avg(np.array(losses), 7)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(steps, losses, color="#7C3AED", lw=1.2, alpha=0.35, label="Raw loss")
ax.plot(steps, sm,     color="#06B6D4", lw=2.5, label="Smoothed (MA-7)")
ax.fill_between(steps, sm - 0.03, sm + 0.03, alpha=0.12, color="#06B6D4")
ax.set(title="SFT Training Loss - Qwen2.5-1.5B-Instruct on LexCrisis Trajectories",
       xlabel="Step", ylabel="Cross-Entropy Loss")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("assets/training_loss_real.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Final training loss: {losses[-1]:.4f}")
print(f"Min training loss:   {min(losses):.4f}")

## 8. Save the Fine-tuned Adapter

In [ ]:
# Save LoRA adapter locally
model.save_pretrained("outputs/lexcrisis-qwen-lora")
tokenizer.save_pretrained("outputs/lexcrisis-qwen-lora")
print("Adapter saved to outputs/lexcrisis-qwen-lora/")

# Optional: push merged model to HF Hub
# from huggingface_hub import login
# login(token="hf_...")
# model.push_to_hub_merged(f"{HF_USERNAME}/lexcrisis-qwen-1.5b-sft", tokenizer, save_method="merged_16bit")

## 9. Phase 3 — Failure Analysis (LLM-as-Judge)

In [ ]:
# Run deterministic failure analysis — no LLM needed, uses the graders directly
result = subprocess.run(
    ["python", "self_improve.py", "--phase", "judge", "--threshold", "0.99"],
    capture_output=True, text=True
)
print(result.stderr)

analyses_path = Path("outputs/self_improve/failure_analyses.json")
if analyses_path.exists():
    analyses = json.loads(analyses_path.read_text())
    print(f"\nFailure analyses produced: {len(analyses)}")
    for a in analyses:
        print(f"  [{a['task_id']}] score={a['final_score']:.4f}: {a['diagnosis']}")

## 10. Results Summary

In [ ]:
# Artifact-backed comparison
from pathlib import Path
import subprocess

subprocess.run([
    "python", "evaluate_runs.py",
    "--run", "oracle=scripted",
    "--run", "base=trace_dir:outputs/policies/base",
    "--run", "sft=trace_dir:outputs/policies/sft",
], check=True)
subprocess.run(["python", "generate_plots.py"], check=True)

summary_path = Path("outputs/evals/summary.json")
print(summary_path.read_text()[:2000])
print("\nGenerated artifact-backed plots in assets/.")
# Close W&B run
if USE_WANDB and wandb.run:
    wandb.finish()
    print('W&B run finished. View at:', wandb.run.url if wandb.run else 'N/A')


---

## Summary

The LexCrisis fine-tuning pipeline is intended to demonstrate three compounding improvements:

1. **SFT** teaches the base model the JSON action schema and environment workflow from oracle-reference trajectories
2. **Seeded curriculum** (deadline randomisation and item shuffling) prevents overfitting to the fixed reference sequence
3. **Artifact-backed evaluation** compares oracle, base, SFT, and optional RL runs using reproducible traces and verifier metrics

The environment's **dense shaped reward** (`grader_score_delta + milestone_bonus + penalty`) supports a short GRPO refinement pass after SFT, but the headline evidence should come from real traces and real eval artifacts rather than manual score scaling.

*LexCrisis — OpenEnv Hackathon | Meta × PyTorch × Hugging Face × Scaler School of Technology*